# Sprint 4 — Pipeline RAG e Assistente Conversacional
Neste notebook construímos o RAG sobre a documentação técnica (Sprints 1 e 2) e instanciamos o Assistente Conversacional (LLM).

In [ ]:
!pip install sentence-transformers faiss-cpu langchain langchain-community huggingface_hub -q
import json
import numpy as np
import faiss
from langchain.text_splitter import RecursiveCharacterTextSplitter

## 1. Chunking e Indexação (FAISS)

In [ ]:
corpus_textos = [
    "Instrucoes de instalacao e manutencao do Motor W22 Plus. Os rolamentos devem ser lubrificados a cada 2.000 horas de operacao ou 6 meses. A vibracao maxima permitida eh de 2,8 mm/s RMS conforme ISO 10816. O torque de aperto dos parafusos de fixacao deve ser de 25 N.m. Verifique o alinhamento do acoplamento: desalinhamento angular maximo de 0,1 mm e desalinhamento paralelo maximo de 0,05 mm. Corrente de partida (Ia/In) 6,5x a corrente nominal.",
    "Siemens 1LA7 Series Installation and Maintenance Manual. Insulation resistance must be measured before commissioning. Minimum insulation resistance 100 MOhm at 1000V DC 60 seconds. Regreasing interval 3500 hours for bearings under normal load. Vibration limits per ISO 10816 Class B less than 2.8 mm/s RMS. During overload conditions current may reach 6x rated current. Temperatura maxima do enrolamento 155C Classe F. Verificar folga axial do eixo maximo 0,3 mm."
]
text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
chunks = []
for doc in corpus_textos:
    chunks.extend(text_splitter.split_text(doc))

from sentence_transformers import SentenceTransformer
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(chunks, convert_to_numpy=True, normalize_embeddings=True)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"Indexado {index.ntotal} vetores no FAISS.")

## 2. LLM e Integração de Contexto (Assistente RAG)

In [ ]:
def buscar_contexto(query, top_k=3):
    q_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    distances, indices = index.search(q_emb, top_k)
    return " ".join([chunks[i] for i in indices[0]])

# Exemplo simulado - em prod usariamos openai ou unsloth localmente
def simular_assistente(query, alerta_contexto=""):
    contexto_recuperado = buscar_contexto(query)
    prompt = f"""
    [PERSONA]: Assistente técnico especialista em motores elétricos.
    [ESTADO ATUAL]: {alerta_contexto}
    [MANUAL]: {contexto_recuperado}
    [PERGUNTA]: {query}
    """
    print("--- PROMPT ---")
    print(prompt)
    return "Resposta do LLM: Com base no manual, lubrificar a cada 2000 horas."

alerta_atual = "ALERTA CRÍTICO: Temperatura atingiu 47°C e vibração 0.47g."
resposta = simular_assistente("Quando devo lubrificar o motor W22?", alerta_atual)